# FoodLens – EfficientNetB0 Full Food-101 Training

This notebook trains **EfficientNetB0** on the complete **Food-101 dataset** (101 classes) using transfer learning with ImageNet weights.

## Priorities (in order)
1. Fast inference speed
2. Low latency API responses
3. Small model size
4. Low memory consumption
5. Easy deployment on Render/Railway/Docker
6. Scalability for concurrent users
7. Good classification accuracy

## Architecture
- **Backbone**: EfficientNetB0 (ImageNet pretrained)
- **Head**: GlobalAveragePooling2D → Dense(128, ReLU) → Dropout(0.3) → Dense(101, Softmax)
- **Training**: 2-phase transfer learning (frozen → fine-tuned)

## Outputs
- `best_model.keras` – best checkpoint by validation accuracy
- `food101_efficientnetb0.h5` – final model
- `confusion_matrix.png` – confusion matrix heatmap
- `training_curves.png` – accuracy/loss plots
- `classification_report.txt` – precision, recall, F1 per class

---
## 1. Setup & Environment Detection

In [ ]:
import os
import sys
import json
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix

# Environment detection
IS_KAGGLE = os.path.exists('/kaggle/input')
IS_COLAB = os.path.exists('/content')

if IS_KAGGLE:
    DATA_DIR = Path('/kaggle/input/food-101')
    WORK_DIR = Path('/kaggle/working')
    print('Detected: Kaggle')
elif IS_COLAB:
    DATA_DIR = Path('/content/food-101')
    WORK_DIR = Path('/content')
    print('Detected: Colab')
else:
    DATA_DIR = Path('./food-101')
    WORK_DIR = Path('.')
    print('Detected: Local')

MODEL_DIR = WORK_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# GPU check
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {len(gpus)}')
if gpus:
    print(f'GPU: {gpus[0].name}')
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# Mixed precision for speed (T4/A100)
policy = tf.keras.mixed_precision.Policy('mixed_float16')
tf.keras.mixed_precision.set_global_policy(policy)
print(f'Compute dtype: {policy.compute_dtype}')
print(f'Variable dtype: {policy.variable_dtype}')

---
## 2. Download Food-101 Dataset

In [ ]:
if not (DATA_DIR / 'images').exists():
    print('Downloading Food-101...')
    !wget -q http://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/food-101.tar.gz -O food-101.tar.gz
    !tar -xzf food-101.tar.gz
    !rm food-101.tar.gz
    print('Download complete!')
else:
    print('Food-101 already exists')

# Verify structure
images_dir = DATA_DIR / 'images'
meta_dir = DATA_DIR / 'meta'
classes = sorted([d.name for d in images_dir.iterdir() if d.is_dir()])
print(f'Number of classes: {len(classes)}')
print(f'First 10 classes: {classes[:10]}')

---
## 3. Data Preprocessing & Augmentation

In [ ]:
# Hyperparameters
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_CLASSES = 101
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Data augmentation
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),
], name='data_augmentation')

# Load train/test splits
def load_split(split_file):
    """Load image paths and labels from split file."""
    paths = []
    labels = []
    with open(meta_dir / f'{split_file}.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            class_name = line.split('/')[0]
            img_path = images_dir / f'{line}.jpg'
            if img_path.exists():
                paths.append(str(img_path))
                labels.append(class_name)
    return paths, labels

train_paths, train_labels = load_split('train')
test_paths, test_labels = load_split('test')

# Create class mapping
class_names = sorted(list(set(train_labels)))
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
idx_to_class = {idx: name for name, idx in class_to_idx.items()}

train_labels_idx = [class_to_idx[label] for label in train_labels]
test_labels_idx = [class_to_idx[label] for label in test_labels]

# Split train into train/validation (85/15)
indices = list(range(len(train_paths)))
random.shuffle(indices)
split_idx = int(0.85 * len(indices))

val_indices = indices[split_idx:]
train_indices = indices[:split_idx]

val_paths = [train_paths[i] for i in val_indices]
val_labels_idx = [train_labels_idx[i] for i in val_indices]

train_paths = [train_paths[i] for i in train_indices]
train_labels_idx = [train_labels_idx[i] for i in train_indices]

print(f'Train: {len(train_paths)} images')
print(f'Validation: {len(val_paths)} images')
print(f'Test: {len(test_paths)} images')
print(f'Classes: {NUM_CLASSES}')

In [ ]:
def load_and_preprocess(path, label):
    """Load, resize, and normalize image."""
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    return img, label

def create_dataset(paths, labels, augment=False, cache=True):
    """Create tf.data.Dataset with optional augmentation."""
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                   num_parallel_calls=tf.data.AUTOTUNE)
    
    if cache:
        ds = ds.cache()
    
    ds = ds.shuffle(buffer_size=1000, seed=SEED)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_dataset(train_paths, train_labels_idx, augment=True)
val_ds = create_dataset(val_paths, val_labels_idx, augment=False)
test_ds = create_dataset(test_paths, test_labels_idx, augment=False)

print('Datasets created!')
print(f'Train batches: {len(train_ds)}')
print(f'Val batches: {len(val_ds)}')
print(f'Test batches: {len(test_ds)}')

---
## 4. Build EfficientNetB0 Model

In [ ]:
def build_model(num_classes=101):
    """Build EfficientNetB0 with custom classifier head."""
    # Load pretrained backbone
    base_model = keras.applications.EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base_model.trainable = False  # Freeze for Phase 1
    
    # Build model
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dense(128, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    
    model = keras.Model(inputs, outputs, name='EfficientNetB0_Food101')
    return model, base_model

model, base_model = build_model(NUM_CLASSES)
model.summary()

# Save model diagram
try:
    keras.utils.plot_model(model, to_file=str(MODEL_DIR / 'model_diagram.png'),
                           show_shapes=True, show_layer_names=True)
    print('Model diagram saved!')
except Exception as e:
    print(f'Could not save diagram: {e}')

---
## 5. Phase 1: Train Classifier Head (Frozen Backbone)

In [ ]:
# Compile for Phase 1
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
callbacks_phase1 = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / 'best_phase1.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print('Phase 1: Training classifier head (backbone frozen)...')
print(f'Epochs: 10 (EarlyStopping patience=3) | LR: 1e-3 | Backbone: Frozen')

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks_phase1,
    verbose=1
)

print(f'Phase 1 complete. Best val accuracy: {max(history_phase1.history["val_accuracy"]):.4f}')

---
## 6. Phase 2: Fine-Tune Top Layers

In [ ]:
# Unfreeze top layers of backbone
base_model.trainable = True

# Freeze all layers except last 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Count trainable parameters
trainable_count = sum([tf.size(v).numpy() for v in model.trainable_variables])
print(f'Trainable parameters: {trainable_count:,}')

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
callbacks_phase2 = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=2,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / 'best_model.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=1,
        min_lr=1e-7,
        verbose=1
    )
]

print('\nPhase 2: Fine-tuning top layers...')
print(f'Epochs: 10 (EarlyStopping patience=3) | LR: 1e-5 | Backbone: Unfrozen (last 20 layers)')

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks_phase2,
    verbose=1
)

print(f'Phase 2 complete. Best val accuracy: {max(history_phase2.history["val_accuracy"]):.4f}')

---
## 7. Evaluation on Test Set

In [ ]:
# Load best model
model = keras.models.load_model(str(MODEL_DIR / 'best_model.keras'))
print('Best model loaded!')

# Evaluate on test set
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f'\nTest Accuracy: {test_acc:.4f}')
print(f'Test Loss: {test_loss:.4f}')

# Get predictions
y_true = []
y_pred = []
for batch in test_ds:
    images, labels = batch
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

In [ ]:
# Classification report
report = classification_report(
    y_true, y_pred,
    target_names=class_names,
    digits=4
)
print(report)

# Save report
with open(WORK_DIR / 'classification_report.txt', 'w') as f:
    f.write(report)
print('Classification report saved!')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

# Plot confusion matrix
plt.figure(figsize=(20, 20))
sns.heatmap(
    cm, annot=False, fmt='d', cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title('Confusion Matrix - EfficientNetB0 on Food-101', fontsize=16)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(WORK_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved!')

---
## 8. Training Curves

In [ ]:
# Combine history from both phases
total_epochs = len(history_phase1.history['accuracy']) + len(history_phase2.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history_phase1.history['accuracy'], label='Phase 1 Train', color='blue')
axes[0].plot(history_phase1.history['val_accuracy'], label='Phase 1 Val', color='cyan')
axes[0].plot(range(len(history_phase1.history['accuracy']), total_epochs),
             history_phase2.history['accuracy'], label='Phase 2 Train', color='green')
axes[0].plot(range(len(history_phase1.history['val_accuracy']), total_epochs),
             history_phase2.history['val_accuracy'], label='Phase 2 Val', color='lime')
axes[0].set_title('Model Accuracy', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history_phase1.history['loss'], label='Phase 1 Train', color='blue')
axes[1].plot(history_phase1.history['val_loss'], label='Phase 1 Val', color='cyan')
axes[1].plot(range(len(history_phase1.history['loss']), total_epochs),
             history_phase2.history['loss'], label='Phase 2 Train', color='green')
axes[1].plot(range(len(history_phase1.history['val_loss']), total_epochs),
             history_phase2.history['val_loss'], label='Phase 2 Val', color='lime')
axes[1].set_title('Model Loss', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(WORK_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved!')

---
## 9. Model Export & Size

In [ ]:
# Save final model
model.save(str(WORK_DIR / 'food101_efficientnetb0.keras'))
model.save(str(WORK_DIR / 'food101_efficientnetb0.h5'))

# Calculate model size
keras_path = WORK_DIR / 'food101_efficientnetb0.keras'
h5_path = WORK_DIR / 'food101_efficientnetb0.h5'

if keras_path.exists():
    print(f'.keras model size: {keras_path.stat().st_size / 1024 / 1024:.2f} MB')
if h5_path.exists():
    print(f'.h5 model size: {h5_path.stat().st_size / 1024 / 1024:.2f} MB')

# Parameter count
total_params = model.count_params()
trainable_params = sum([tf.size(v).numpy() for v in model.trainable_variables])
print(f'\nTotal parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

In [ ]:
# Inference speed benchmark
import time

# Warm up
dummy = np.random.rand(1, IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
for _ in range(5):
    model.predict(dummy, verbose=0)

# Benchmark
times = []
for _ in range(50):
    start = time.time()
    model.predict(dummy, verbose=0)
    times.append(time.time() - start)

avg_time = np.mean(times) * 1000  # ms
print(f'Inference speed: {avg_time:.2f} ms/image')
print(f'Throughput: {1000/avg_time:.1f} images/sec')

---
## 10. Summary & Results

In [ ]:
# Save results summary
results = {
    'model': 'EfficientNetB0',
    'dataset': 'Food-101 (101 classes)',
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'total_epochs': total_epochs,
    'test_accuracy': float(test_acc),
    'test_loss': float(test_loss),
    'model_size_mb': round(h5_path.stat().st_size / 1024 / 1024, 2) if h5_path.exists() else 0,
    'inference_time_ms': round(avg_time, 2),
    'total_params': total_params,
    'trainable_params': int(trainable_params),
    'timestamp': datetime.now().isoformat()
}

with open(WORK_DIR / 'training_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('\n' + '='*60)
print('TRAINING COMPLETE')
print('='*60)
print(f'Model: EfficientNetB0')
print(f'Dataset: Food-101 ({len(class_names)} classes)')
print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test Loss: {test_loss:.4f}')
print(f'Model Size: {results["model_size_mb"]:.2f} MB')
print(f'Inference Speed: {avg_time:.2f} ms/image')
print('='*60)
print('\nSaved files:')
for f in WORK_DIR.iterdir():
    if f.suffix in ['.keras', '.h5', '.png', '.txt', '.json']:
        print(f'  - {f.name}')

---
## 11. Download Model (Colab/Kaggle)

If running on Colab or Kaggle, download the model files to your local machine.

In [ ]:
if IS_COLAB:
    from google.colab import files
    print('Downloading model files...')
    files.download(str(WORK_DIR / 'food101_efficientnetb0.h5'))
    files.download(str(WORK_DIR / 'classification_report.txt'))
    files.download(str(WORK_DIR / 'training_results.json'))
    print('Download complete!')
elif IS_KAGGLE:
    print('Model saved to /kaggle/working/')
    print('Click the folder icon on the right to download files.')
else:
    print(f'Models saved in: {WORK_DIR}')